# Kim Konuşuyor? — İki modelin eğitimi ve karşılaştırması

Bu notebook iki konuşmacı tanıma modelini **aynı data** ile eğitir ve karşılaştırır:

- **Model A:** sıfırdan eğitilen küçük bir CNN (sadece sizin 4 kişinin sesini duyar)
- **Model B:** önceden binlerce kişiyle eğitilmiş ECAPA + üstünde küçük bir sınıflandırıcı

**Nasıl çalıştırılır:** sağ üstten kernel olarak `.venv`'i seç, sonra hücreleri yukarıdan aşağıya **Shift + Enter** ile sırayla çalıştır.
Data hazırlığı (`scripts/asama2.py`) önceden çalıştırılmış olmalı.

## 0. Kurulum

Kütüphaneleri yükler, 48 CPU çekirdeğinin hepsini kullanmasını söyler ve parçaların listesini (`manifest.csv`) okur.

**Çıktıda bak:** kişi başı train / val / test parça sayıları. Train sayıları temiz parça + 3 augmentation kopyasını içerir.

In [ ]:
import copy, json, sys, time
from pathlib import Path

sys.path.insert(0, "scripts")
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Audio, display

from ortak import SR, oku, yollar
from modeller import (diarize, embedding_cikar, ecapa_yukle, kaydet_a, kaydet_b, log_mel, mel_donusumu,
                      model_a_yap, model_b_kafa_yap, tahminci_a, tahminci_b)

torch.set_num_threads(48)
torch.manual_seed(0)
np.random.seed(0)

Y = yollar()
manifest = pd.read_csv(Y["parcalar"] / "manifest.csv")
ISIMLER = sorted(manifest.kisi.unique())
KISI_NO = {k: i for i, k in enumerate(ISIMLER)}
RENK = dict(zip(ISIMLER, ["#7c3aed", "#0d9488", "#f59e0b", "#ef4444", "#3b82f6", "#84cc16"]))

print("Kişiler:", ISIMLER)
manifest.groupby(["kisi", "bolum"]).size().unstack(fill_value=0)[["train", "val", "test"]]

## 1. Her kişiden bir örnek: ses ve spektrogram

Spektrogram sesi bir resme çevirir:
- **yatay eksen:** zaman (2 saniye)
- **dikey eksen:** frekans, yani sesin inceliği (aşağısı kalın, yukarısı ince sesler)
- **renk:** o anda o frekansta sesin ne kadar güçlü olduğu (parlak = güçlü)

**Çıktıda bak:** her kişinin sesini dinle. Spektrogramdaki yatay parlak çizgiler (harmonikler) sık mı, seyrek mi?
Seyrek çizgiler daha ince (yüksek perdeli) bir sese, sık çizgiler daha kalın bir sese işaret eder. Model A'nın "baktığı resim" tam olarak bu.

In [ ]:
mel = mel_donusumu()

for kisi in ISIMLER:
    satir = manifest[(manifest.kisi == kisi) & (manifest.bolum == "train") & (manifest.aug == "temiz")].iloc[10]
    x = oku(Y["parcalar"] / satir.yol)
    print(f"{kisi}  ({satir.yol})")
    display(Audio(x, rate=SR))
    S = log_mel(mel, torch.from_numpy(x)[None])[0, 0].numpy()
    fig, ax = plt.subplots(figsize=(7, 2.2))
    ax.imshow(S, origin="lower", aspect="auto", cmap="magma", extent=[0, 2, 0, 80])
    ax.set_title(kisi); ax.set_xlabel("zaman (sn)"); ax.set_ylabel("mel bandı")
    plt.show()

## 2. Augmentation örnekleri

Aşama 2'de her train parçasının yanına 3 değiştirilmiş kopya yazıldı. Böylece model aynı sesi farklı koşullarda da duyar:
- **gürültü:** ofisin oda sesi eklendi
- **yankı:** sanki daha büyük, yankılı bir odada konuşulmuş gibi
- **seviye:** ses daha yüksek ya da daha alçak

Hızlandırma/yavaşlatma yok: sesin perdesini değiştirir ve kişi başka biri gibi duyulur.

**Çıktıda bak:** dört versiyonu dinle. Hepsinde aynı kişiyi tanıyabiliyor musun? Model de tanıyabilmeli.

In [ ]:
ornek = manifest[(manifest.bolum == "train") & (manifest.aug == "temiz")].iloc[100].yol
kok = Y["parcalar"] / ornek
surumler = {"temiz": kok, "gürültü": kok.with_name(kok.stem + "_gurultu.wav"),
            "yankı": kok.with_name(kok.stem + "_yanki.wav"), "seviye": kok.with_name(kok.stem + "_seviye.wav")}

fig, eksenler = plt.subplots(1, 4, figsize=(14, 2.6), sharey=True)
for ax, (ad, yol) in zip(eksenler, surumler.items()):
    S = log_mel(mel, torch.from_numpy(oku(yol))[None])[0, 0].numpy()
    ax.imshow(S, origin="lower", aspect="auto", cmap="magma", extent=[0, 2, 0, 80], vmin=-14, vmax=4)
    ax.set_title(ad); ax.set_xlabel("zaman (sn)")
plt.suptitle(ornek.split("/")[1]); plt.tight_layout(); plt.show()
for ad, yol in surumler.items():
    print(ad); display(Audio(oku(yol), rate=SR))

## 3. Datayı hafızaya yükleme

Bütün parçalar bir kerede okunur ve Model A için spektrograma çevrilir (birkaç saniye).
`targets` listesi ve `to_rgb` fonksiyonu canlı panel için: panel her kişiden bir örnek seçer ve spektrogramı magma renkleriyle çizer.

**Çıktıda bak:** üç bölümün boyutları. Train'de kişi başına ~4 kat parça var (temiz + 3 kopya).

In [ ]:
def yukle(bolum):
    s = manifest[manifest.bolum == bolum]
    wav = torch.from_numpy(np.stack([oku(Y["parcalar"] / p) for p in s.yol]))
    return wav, torch.tensor([KISI_NO[k] for k in s.kisi])

class SpektrogramDataset(torch.utils.data.Dataset):
    def __init__(self, wav, y):
        with torch.no_grad():
            self.x = torch.cat([log_mel(mel, wav[i:i + 256]) for i in range(0, len(wav), 256)])
        self.y = y
        self.targets = y.tolist()  # canlı panel her kişiden bir örnek bulmak için kullanır

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.x[i], int(self.y[i])

_magma = matplotlib.colormaps["magma"]
def to_rgb(t):
    """(1, mel, zaman) spektrogram -> (H, W, 3), 0-1 arası renkli resim (kalın sesler altta)."""
    s = t[0]
    s = (s - s.min()) / (s.max() - s.min() + 1e-8)
    return torch.from_numpy(_magma(s.flip(0).numpy())[..., :3]).float()

t0 = time.time()
wav_tr, y_tr = yukle("train")
wav_va, y_va = yukle("val")
wav_te, y_te = yukle("test")
ds_tr, ds_va, ds_te = (SpektrogramDataset(w, y) for w, y in ((wav_tr, y_tr), (wav_va, y_va), (wav_te, y_te)))
print(f"{time.time() - t0:.1f} sn")
for ad, ds in (("train", ds_tr), ("val", ds_va), ("test", ds_te)):
    print(f"{ad:5}: {len(ds):5} parça, spektrogram boyutu {tuple(ds.x.shape[1:])}")
eksik = [k for k in ISIMLER if KISI_NO[k] not in set(ds_va.targets)]
if eksik:
    print("Dikkat: validation'da olmayan kişi(ler):", eksik, "-> validation sonuçları bu kişiyi ölçmüyor.")

## 4. Model A — sıfırdan CNN

5 blok, her biri: **Conv 3×3** (küçük bir damga spektrogramın üstünden geçer) → **BatchNorm** → **ReLU** → **MaxPool** (resim yarıya küçülür).
Sonra **global average pooling** tüm resmin ortalamasını alır: model sesin parçanın *neresinde* olduğunu değil, *nasıl* bir ses olduğunu öğrenir.
Son katman 128 sayıyı kişi sayısı kadar puana çevirir.

**Çıktıda bak:** her bloktan sonra resim nasıl küçülüyor ve kanal (özellik) sayısı nasıl artıyor; toplam parametre sayısı.

In [ ]:
torch.manual_seed(0)
model_a = model_a_yap(len(ISIMLER))
h = ds_tr.x[:1]
print("girdi           ", tuple(h.shape[1:]))
for i, b in enumerate(model_a[:5], 1):
    h = b(h)
    print(f"blok {i} çıkışı    {tuple(h.shape[1:])}")
print(f"\nToplam parametre: {sum(p.numel() for p in model_a.parameters()):,}")
model_a

## 5. Model A'nın eğitimi (canlı panel)

Model rastgele ağırlıklarla başlar ve sadece sizin seslerinizden öğrenir. Her epoch'ta train parçalarının hepsini bir kez görür.
Her epoch sonunda **validation** (oturum 4) üzerinde ölçülür; validation loss'u en iyi olan epoch'un ağırlıkları saklanır.

**Canlı panel:** hücre çalışınca çıktıda `Canlı panel: http://localhost:8002/` yazar. Tarayıcıda yeni bir sekmede
**https://student.exposureai.org/lab/proxy/8002/** adresini aç (port farklıysa 8002'yi onunla değiştir, sondaki `/` kalsın).

**Çıktıda bak:** her epoch'ta train ve val accuracy. Train çok yüksek ama val düşük kalıyorsa model kayıtları ezberliyor demektir (overfitting).
Yaklaşık 3 dakika sürer.

In [ ]:
from live import CNNLive

EPOCH_A = 40

def degerlendir(model, ds):
    model.eval()
    with torch.no_grad():
        out = torch.cat([model(ds.x[i:i + 256]) for i in range(0, len(ds), 256)])
    return F.cross_entropy(out, ds.y).item(), (out.argmax(1) == ds.y).float().mean().item()

def egit(model, epochs, lr, view=None, weight_decay=1e-4, batch=64):
    # Data zaten hafızada (spektrogram tensörleri): ekstra worker'a gerek yok
    dl = torch.utils.data.DataLoader(ds_tr, batch_size=batch, shuffle=True, num_workers=0)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * len(dl))
    history, en_iyi = [], (float("inf"), None, 0)
    for ep in range(1, epochs + 1):
        model.train()
        top_kayip = dogru = n = 0
        for i, (x, y) in enumerate(dl, start=1):
            out = model(x)
            loss = F.cross_entropy(out, y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            top_kayip += loss.item() * len(y); dogru += (out.argmax(1) == y).sum().item(); n += len(y)
            if view: view(model, history, i, len(dl), x=x, y=y)
        va_loss, va_acc = degerlendir(model, ds_va)
        history.append(dict(epoch=ep, train_loss=top_kayip / n, train_acc=dogru / n, val_loss=va_loss, val_acc=va_acc))
        if va_loss < en_iyi[0]:
            en_iyi = (va_loss, copy.deepcopy(model.state_dict()), ep)
        if view: view(model, history, len(dl), len(dl), force=True)
        print(f"epoch {ep:2d}   train loss {top_kayip / n:.3f}  acc {dogru / n:6.1%}   |   val loss {va_loss:.3f}  acc {va_acc:6.1%}")
    model.load_state_dict(en_iyi[1])
    print(f"\nEn iyi validation loss: epoch {en_iyi[2]}, bu epoch'un ağırlıkları kullanılıyor.")
    return history

view_a = CNNLive(model_a, ds_tr, to_rgb, ISIMLER, epochs=EPOCH_A, window=False)
t0 = time.time()
hist_a = egit(model_a, EPOCH_A, 1e-3, view=view_a)
sure_a = time.time() - t0
view_a.close()
print(f"Model A eğitimi: {sure_a / 60:.1f} dakika")

## 6. Model A'nın eğitim grafiği

**Çıktıda bak:** iki çizgi (train ve validation) birlikte iniyor/çıkıyorsa model genelliyor.
Train loss inmeye devam ederken validation loss yükselmeye başlıyorsa, o noktadan sonra model ezberliyor.

In [ ]:
def gecmis_ciz(h, baslik):
    h = pd.DataFrame(h)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
    a1.plot(h.epoch, h.train_loss, label="train"); a1.plot(h.epoch, h.val_loss, label="validation")
    a1.set_title(f"{baslik} — loss"); a1.set_xlabel("epoch"); a1.legend()
    a2.plot(h.epoch, h.train_acc, label="train"); a2.plot(h.epoch, h.val_acc, label="validation")
    a2.set_title(f"{baslik} — accuracy"); a2.set_xlabel("epoch"); a2.set_ylim(0, 1.02); a2.legend()
    plt.tight_layout(); plt.show()

gecmis_ciz(hist_a, "Model A (sıfırdan CNN)")

## 7. Model B — ECAPA embedding'leri

ECAPA (VoxCeleb'de 7.000'den fazla kişiyle önceden eğitilmiş) **donuk** kalır: ağırlıkları hiç değişmez.
Her 2 saniyelik parçayı 192 sayılık bir "ses parmak izine" (embedding) çevirir. Bu sayılar bir kez hesaplanıp diske kaydedilir;
notebook'u tekrar çalıştırırsan diskten okunur.

**Çıktıda bak:** her bölümün embedding boyutu (parça sayısı × 192). İlk seferde ~1–2 dakika sürer.

In [ ]:
ecapa = ecapa_yukle()
print(f"ECAPA: {sum(p.numel() for p in ecapa.mods.embedding_model.parameters()) / 1e6:.1f} M parametre, hepsi donuk")

onbellek = Y["parcalar"].parent / "embeddings.pt"
imza = (len(wav_tr), len(wav_va), len(wav_te), float(wav_tr[:, :100].sum()))
if onbellek.exists() and torch.load(onbellek).get("imza") == imza:
    d = torch.load(onbellek)
    emb_tr, emb_va, emb_te = d["tr"], d["va"], d["te"]
    print("Embedding'ler diskten okundu.")
else:
    t0 = time.time()
    emb_tr, emb_va, emb_te = (embedding_cikar(ecapa, w) for w in (wav_tr, wav_va, wav_te))
    torch.save({"imza": imza, "tr": emb_tr, "va": emb_va, "te": emb_te}, onbellek)
    print(f"Embedding'ler hesaplandı: {time.time() - t0:.0f} sn")
print("train", tuple(emb_tr.shape), " val", tuple(emb_va.shape), " test", tuple(emb_te.shape))

## 8. Model B'nin sınıflandırıcısı

ECAPA'nın üstüne sadece **bir Linear katman** (192 sayı → kişi sayısı kadar puan) eğitilir. Embedding önce uzunluğu 1 olacak şekilde normalize edilir (L2).
Data çok küçük (sadece 192 sayılık vektörler) olduğu için bütün train seti tek seferde verilir; birkaç saniye sürer. Canlı panel yok: öğrenen kısım bu kadar küçük.
Pretraining'in farkı da tam olarak bu: zor iş (sesi dinleyip parmak izini çıkarmak) zaten yapılmış.

**Çıktıda bak:** Model A'ya göre ne kadar hızlı ve hangi doğruluğa ulaşıyor.

In [ ]:
torch.manual_seed(0)
kafa = model_b_kafa_yap(len(ISIMLER))
X_tr, X_va, X_te = (F.normalize(e, dim=-1) for e in (emb_tr, emb_va, emb_te))
opt = torch.optim.AdamW(kafa.parameters(), lr=0.05, weight_decay=1e-3)
EPOCH_B = 300

hist_b, en_iyi = [], (float("inf"), None, 0)
t0 = time.time()
for ep in range(1, EPOCH_B + 1):
    kafa.train()
    out = kafa(X_tr)
    loss = F.cross_entropy(out, y_tr)
    opt.zero_grad(); loss.backward(); opt.step()
    kafa.eval()
    with torch.no_grad():
        out_va = kafa(X_va)
        va_loss = F.cross_entropy(out_va, y_va).item()
    hist_b.append(dict(epoch=ep, train_loss=loss.item(), train_acc=(out.argmax(1) == y_tr).float().mean().item(),
                       val_loss=va_loss, val_acc=(out_va.argmax(1) == y_va).float().mean().item()))
    if va_loss < en_iyi[0]:
        en_iyi = (va_loss, copy.deepcopy(kafa.state_dict()), ep)
    if ep % 50 == 0 or ep == 1:
        h = hist_b[-1]
        print(f"adım {ep:3d}   train loss {h['train_loss']:.3f}  acc {h['train_acc']:6.1%}   |   val loss {h['val_loss']:.3f}  acc {h['val_acc']:6.1%}")
kafa.load_state_dict(en_iyi[1])
sure_b_kafa = time.time() - t0
print(f"\nEn iyi validation loss: adım {en_iyi[2]}. Sınıflandırıcının eğitimi: {sure_b_kafa:.1f} sn")

## 9. Model B'nin eğitim grafiği

Burada her "epoch" bütün train setinin tek seferde verildiği bir adım.

**Çıktıda bak:** Model A'nın grafiğiyle karşılaştır: çizgiler ne kadar hızlı yükseliyor, train ile validation arasında ne kadar boşluk var?

In [ ]:
gecmis_ciz(hist_b, "Model B (ECAPA + Linear)")

## 10. Test parçalarında karışıklık matrisi

Test = oturum 3 (laptop, orta mesafe). Bu kayıtlar eğitimde hiç kullanılmadı; demo da böyle yapılacak.
Her satır gerçek kişi, her sütun modelin tahmini. Köşegen (sol üstten sağ alta) doğru tahminler; köşegen dışındaki her kutu bir karışıklık.

**Çıktıda bak:** model kimi kiminle karıştırıyor? Sesleri birbirine benzeyen kişilere özellikle bak.

In [ ]:
with torch.no_grad():
    p_te_a = tahminci_a(model_a, mel)(wav_te)
    p_te_b = torch.softmax(kafa(X_te), -1)

def karisiklik(y, p):
    m = np.zeros((len(ISIMLER), len(ISIMLER)), dtype=int)
    for a, b in zip(y.tolist(), p.argmax(1).tolist()):
        m[a, b] += 1
    return m

fig, eksenler = plt.subplots(1, 2, figsize=(12, 5))
test_acc = {}
for ax, (ad, p) in zip(eksenler, (("Model A (sıfırdan CNN)", p_te_a), ("Model B (ECAPA)", p_te_b))):
    m = karisiklik(y_te, p)
    oran = m / m.sum(1, keepdims=True)
    ax.imshow(oran, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(ISIMLER)):
        for j in range(len(ISIMLER)):
            ax.text(j, i, f"{m[i, j]}\n%{100 * oran[i, j]:.0f}", ha="center", va="center",
                    color="white" if oran[i, j] > 0.5 else "black", fontsize=9)
    ax.set_xticks(range(len(ISIMLER)), ISIMLER); ax.set_yticks(range(len(ISIMLER)), ISIMLER)
    ax.set_xlabel("modelin tahmini"); ax.set_ylabel("gerçek kişi")
    test_acc[ad] = np.trace(m) / m.sum()
    ax.set_title(f"{ad}\ntest accuracy %{100 * test_acc[ad]:.1f}")
plt.tight_layout(); plt.show()

## 11. Sahte sohbetlerde DER

20 sahte sohbetin her birinde:
1. **Silero VAD** konuşmanın nerede olduğunu bulur (iki model için aynı).
2. Model sohbeti **2 saniyelik pencereyle, 0,5 saniye kaydırarak** dinler.
3. Her 0,5 saniyelik dilimin kişisi, üstüne düşen pencerelerin olasılık ortalamasıyla seçilir; art arda aynı kişi birleşir.
4. Sonuç, doğru cevap dosyasıyla (RTTM) karşılaştırılır.

**DER** = (yanlış kişi + kaçırılan konuşma + yanlış alarm) / toplam konuşma süresi. 0'a ne kadar yakınsa o kadar iyi.
İsimler belli olduğu için hesabı `pyannote.metrics`'teki `IdentificationErrorRate` yapar.

**Çıktıda bak:** iki model aynı VAD'ı kullandığı için kaçırılan konuşma ve yanlış alarm ikisinde de aynı; aradaki farkı **yanlış kişi** belirler. ~2 dakika sürer.

In [ ]:
from pyannote.core import Annotation, Segment
from pyannote.metrics.identification import IdentificationErrorRate
from silero_vad import load_silero_vad

vad = load_silero_vad()
tahminciler = {"Model A": tahminci_a(model_a, mel), "Model B": tahminci_b(ecapa, kafa)}

def rttm_oku(yol):
    ref = Annotation()
    for satir in open(yol):
        p = satir.split()
        ref[Segment(float(p[3]), float(p[3]) + float(p[4]))] = p[7]
    return ref

def annotation(parcalar):
    a = Annotation()
    for b, e, k, _ in parcalar:
        a[Segment(b, e)] = k
    return a

sohbetler = sorted(Y["sohbetler"].glob("sohbet_*.wav"))
sonuc = {ad: {"parcalar": {}, "bilesen": dict.fromkeys(["total", "confusion", "missed detection", "false alarm"], 0.0)}
         for ad in tahminciler}
t0 = time.time()
for wav_yol in sohbetler:
    x = oku(wav_yol)
    ref = rttm_oku(wav_yol.with_suffix(".rttm"))
    for ad, fn in tahminciler.items():
        parcalar = diarize(x, fn, vad, ISIMLER)
        sonuc[ad]["parcalar"][wav_yol.stem] = parcalar
        d = IdentificationErrorRate()(ref, annotation(parcalar), detailed=True)
        for k in sonuc[ad]["bilesen"]:
            sonuc[ad]["bilesen"][k] += d[k]
print(f"{len(sohbetler)} sohbet, {time.time() - t0:.0f} sn\n")

satirlar = []
for ad, s in sonuc.items():
    b = s["bilesen"]
    s["der"] = (b["confusion"] + b["missed detection"] + b["false alarm"]) / b["total"]
    satirlar.append({"model": ad, "DER": f"%{100 * s['der']:.1f}", "yanlış kişi": f"%{100 * b['confusion'] / b['total']:.1f}",
                     "kaçırılan konuşma": f"%{100 * b['missed detection'] / b['total']:.1f}",
                     "yanlış alarm": f"%{100 * b['false alarm'] / b['total']:.1f}"})
pd.DataFrame(satirlar).set_index("model")

## 12. Bir sohbetin zaman çizelgesi

Aynı sohbet için alt alta: **doğru cevap**, **Model A** ve **Model B**. Her kişi kendi renginde, her bölümün üstünde isim ve modelin ne kadar emin olduğu (% confidence).

**Çıktıda bak:** modeller konuşmacı değişimlerini doğru saniyede yakalıyor mu? Yanlış renkli bölümler nerede ve ne kadar eminken yanılmış?

In [ ]:
SOHBET = "sohbet_01"
ref = rttm_oku(Y["sohbetler"] / f"{SOHBET}.rttm")
satirlar = [("Doğru cevap", [(s.start, s.end, k, None) for s, _, k in ref.itertracks(yield_label=True)])]
satirlar += [(f"{ad}  (DER %{100 * sonuc[ad]['der']:.1f})", sonuc[ad]["parcalar"][SOHBET]) for ad in tahminciler]

fig, eksenler = plt.subplots(len(satirlar), 1, figsize=(15, 1.3 * len(satirlar)), sharex=True)
for ax, (baslik, parcalar) in zip(eksenler, satirlar):
    for b, e, k, guven in parcalar:
        ax.broken_barh([(b, e - b)], (0, 1), color=RENK[k])
        if e - b > 1.2:
            ax.text((b + e) / 2, 0.5, k if guven is None else f"{k}\n%{100 * guven:.0f}",
                    ha="center", va="center", color="white", fontsize=8, fontweight="bold")
    ax.set_yticks([]); ax.set_ylabel(baslik, rotation=0, ha="right", va="center")
    ax.set_xlim(0, 60)
eksenler[-1].set_xlabel("zaman (sn)")
plt.tight_layout(); plt.show()
print("Sohbeti dinle:")
display(Audio(oku(Y["sohbetler"] / f"{SOHBET}.wav"), rate=SR))

## 13. Karşılaştırma tablosu

İki model yan yana. **DER** en önemli satır; ama Aşama 5'te canlı testte de bakacağız: rakamlar bir şey söyler, asıl test gerçek hayat.

In [ ]:
def ort_guven(ad):
    tum = [(e - b, g) for p in sonuc[ad]["parcalar"].values() for b, e, _, g in p]
    return sum(s * g for s, g in tum) / sum(s for s, _ in tum)

tablo = pd.DataFrame({
    "Model A (sıfırdan CNN)": {
        "eğitilen parametre": f"{sum(p.numel() for p in model_a.parameters()):,}",
        "eğitim süresi": f"{sure_a / 60:.1f} dk",
        "validation accuracy (seçilen epoch)": f"%{100 * min(hist_a, key=lambda h: h['val_loss'])['val_acc']:.1f}",
        "test accuracy (2 sn parça)": f"%{100 * test_acc['Model A (sıfırdan CNN)']:.1f}",
        "DER (20 sahte sohbet)": f"%{100 * sonuc['Model A']['der']:.1f}",
        "ortalama confidence": f"%{100 * ort_guven('Model A'):.0f}",
    },
    "Model B (ECAPA + Linear)": {
        "eğitilen parametre": f"{sum(p.numel() for p in kafa.parameters()):,}  (+20,8 M donuk)",
        "eğitim süresi": f"{sure_b_kafa:.0f} sn (+ embedding'ler)",
        "validation accuracy (seçilen epoch)": f"%{100 * min(hist_b, key=lambda h: h['val_loss'])['val_acc']:.1f}",
        "test accuracy (2 sn parça)": f"%{100 * test_acc['Model B (ECAPA)']:.1f}",
        "DER (20 sahte sohbet)": f"%{100 * sonuc['Model B']['der']:.1f}",
        "ortalama confidence": f"%{100 * ort_guven('Model B'):.0f}",
    },
})
tablo

## 14. Modelleri kaydetme

İki model de `modeller/` klasörüne `.pt` olarak kaydedilir; Aşama 5'teki Gradio uygulaması bunları yükler.
Model B için sadece küçük sınıflandırıcı kaydedilir; ECAPA'nın kendisi `pretrained_models/` içinde zaten duruyor.
Sonuçlar da `modeller/sonuclar.json`'a yazılır (web uygulamasındaki bilgi kartı için).

In [ ]:
kaydet_a("modeller/model_a.pt", model_a, ISIMLER)
kaydet_b("modeller/model_b_kafa.pt", kafa, ISIMLER)
json.dump({ad: {"der": sonuc[ad]["der"], "bilesenler": sonuc[ad]["bilesen"]} for ad in sonuc} |
          {"test_acc": test_acc, "isimler": ISIMLER},
          open("modeller/sonuclar.json", "w"), ensure_ascii=False, indent=1)
for p in sorted(Path("modeller").iterdir()):
    print(f"{p}  ({p.stat().st_size / 1e3:.0f} KB)")